In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import shutil
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Đã import thư viện thành công!")

✅ Đã import thư viện thành công!


In [3]:
BASE_PATH = "/content/drive/MyDrive/HAM10000/raw"

# Thư mục chứa ảnh gốc (thường là 2 part)
IMAGE_DIR1 = os.path.join(BASE_PATH, "HAM10000_images_part_1")
IMAGE_DIR2 = os.path.join(BASE_PATH, "HAM10000_images_part_2")
METADATA_PATH = os.path.join(BASE_PATH, "HAM10000_metadata.csv")

# Thư mục đích (sẽ tạo folder theo class)
OUTPUT_DIR = os.path.join(BASE_PATH, "organized_by_class")

print("Base path:", BASE_PATH)
print("Metadata:", METADATA_PATH)

Base path: /content/drive/MyDrive/HAM10000/raw
Metadata: /content/drive/MyDrive/HAM10000/raw/HAM10000_metadata.csv


In [4]:
df = pd.read_csv(METADATA_PATH)
print("Shape:", df.shape)
print("\nCác class (dx):")
print(df['dx'].value_counts())

# Mapping tên class đầy đủ (tùy chọn)
dx_mapping = {
    'akiec': 'Actinic_keratoses',
    'bcc': 'Basal_cell_carcinoma',
    'bkl': 'Benign_keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic_nevi',
    'vasc': 'Vascular_lesions'
}

df['dx_full'] = df['dx'].map(dx_mapping)
print("\n✅ Đã load metadata!")

Shape: (10015, 7)

Các class (dx):
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

✅ Đã load metadata!


In [ ]:
# Tạo thư mục output
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Tạo folder cho từng class
classes = df['dx'].unique()
for cls in classes:
    folder_name = dx_mapping.get(cls, cls)
    os.makedirs(os.path.join(OUTPUT_DIR, folder_name), exist_ok=True)

print("✅ Đã tạo các folder theo class:")
for cls in sorted(os.listdir(OUTPUT_DIR)):
    print("   📁", cls)

✅ Đã tạo các folder theo class:
   📁 Actinic_keratoses
   📁 Basal_cell_carcinoma
   📁 Benign_keratosis
   📁 Dermatofibroma
   📁 Melanocytic_nevi
   📁 Melanoma
   📁 Vascular_lesions


In [ ]:
def find_image_path(image_id):
    """Tìm ảnh trong 2 thư mục part 1 hoặc part 2"""
    for img_dir in [IMAGE_DIR1, IMAGE_DIR2]:
        path = os.path.join(img_dir, f"{image_id}.jpg")
        if os.path.exists(path):
            return path
    return None

# Thực hiện copy
copied = 0
missing = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Đang tổ chức ảnh"):
    image_id = row['image_id']
    class_name = dx_mapping.get(row['dx'], row['dx'])

    src_path = find_image_path(image_id)
    if src_path and os.path.exists(src_path):
        dst_path = os.path.join(OUTPUT_DIR, class_name, f"{image_id}.jpg")

        # Copy (bạn có thể đổi thành shutil.move nếu muốn di chuyển)
        shutil.copy2(src_path, dst_path)
        copied += 1
    else:
        missing += 1

print(f"\n✅ Hoàn thành!")
print(f"   Đã copy: {copied} ảnh")
print(f"   Thiếu: {missing} ảnh")

In [ ]:
print("Số lượng ảnh trong từng class:\n")
for class_folder in sorted(os.listdir(OUTPUT_DIR)):
    count = len(os.listdir(os.path.join(OUTPUT_DIR, class_folder)))
    print(f"📁 {class_folder:25} : {count:4} ảnh")

Số lượng ảnh trong từng class:

📁 Actinic_keratoses         :  327 ảnh
📁 Basal_cell_carcinoma      :  514 ảnh
📁 Benign_keratosis          : 1099 ảnh
📁 Dermatofibroma            :  115 ảnh
📁 Melanocytic_nevi          : 6705 ảnh
📁 Melanoma                  : 1113 ảnh
📁 Vascular_lesions          :  142 ảnh


In [7]:
import zipfile
import math
import random # Cần để lấy mẫu ngẫu nhiên

# Thư mục chứa các file zip đã tạo trong môi trường runtime
ZIPPED_OUTPUT_DIR = "/content/zipped"
os.makedirs(ZIPPED_OUTPUT_DIR, exist_ok=True)

IMAGES_PER_CLASS_SAMPLE = 50 # Số lượng ảnh ngẫu nhiên muốn lấy từ mỗi class

print(f"Bắt đầu chọn ngẫu nhiên khoảng {IMAGES_PER_CLASS_SAMPLE} ảnh từ mỗi class để tạo file zip...")

all_sampled_images_for_zip = {} # Lưu trữ các đường dẫn ảnh đã chọn ngẫu nhiên theo từng class

for class_folder_name in sorted(os.listdir(OUTPUT_DIR)):
    class_path = os.path.join(OUTPUT_DIR, class_folder_name)
    if not os.path.isdir(class_path):
        continue

    image_files = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))]
    num_images_in_class = len(image_files)

    if num_images_in_class == 0:
        print(f"   Không có ảnh nào trong thư mục: {class_folder_name}")
        continue

    # Lấy ngẫu nhiên IMAGES_PER_CLASS_SAMPLE ảnh, hoặc ít hơn nếu class đó không đủ
    sampled_for_class = random.sample(image_files, min(IMAGES_PER_CLASS_SAMPLE, num_images_in_class))
    all_sampled_images_for_zip[class_folder_name] = sampled_for_class
    print(f"   ✅ Class '{class_folder_name}': đã chọn {len(sampled_for_class)} ảnh ngẫu nhiên.")


print(f"\nBắt đầu tạo file zip chứa các ảnh mẫu đã chọn...")

# Tên file zip kết hợp các ảnh mẫu từ tất cả các class
combined_zip_file_name = "random_image_samples_by_class.zip"
combined_zip_file_path = os.path.join(ZIPPED_OUTPUT_DIR, combined_zip_file_name)

images_added_count = 0
with zipfile.ZipFile(combined_zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for class_name, img_paths in all_sampled_images_for_zip.items():
        for img_path in img_paths:
            # Đường dẫn của ảnh bên trong file zip: ClassName/image_name.jpg
            arcname = os.path.join(class_name, os.path.basename(img_path))
            zipf.write(img_path, arcname)
            images_added_count += 1

print(f"\n✅ Hoàn thành việc tạo file zip: {combined_zip_file_name}")
print(f"   Tổng số {images_added_count} ảnh đã được thêm vào file zip.")
print(f"Kiểm tra trong thư mục: {ZIPPED_OUTPUT_DIR}")
print("Bạn có thể tải file này về từ Colab runtime (nó sẽ bị xóa khi runtime ngắt kết nối).")

Bắt đầu chọn ngẫu nhiên khoảng 50 ảnh từ mỗi class để tạo file zip...
   ✅ Class 'Actinic_keratoses': đã chọn 50 ảnh ngẫu nhiên.
   ✅ Class 'Basal_cell_carcinoma': đã chọn 50 ảnh ngẫu nhiên.
   ✅ Class 'Benign_keratosis': đã chọn 50 ảnh ngẫu nhiên.
   ✅ Class 'Dermatofibroma': đã chọn 50 ảnh ngẫu nhiên.
   ✅ Class 'Melanocytic_nevi': đã chọn 50 ảnh ngẫu nhiên.
   ✅ Class 'Melanoma': đã chọn 50 ảnh ngẫu nhiên.
   ✅ Class 'Vascular_lesions': đã chọn 50 ảnh ngẫu nhiên.

Bắt đầu tạo file zip chứa các ảnh mẫu đã chọn...

✅ Hoàn thành việc tạo file zip: random_image_samples_by_class.zip
   Tổng số 350 ảnh đã được thêm vào file zip.
Kiểm tra trong thư mục: /content/zipped
Bạn có thể tải file này về từ Colab runtime (nó sẽ bị xóa khi runtime ngắt kết nối).
